# Error analysis and reflection (#18)

Where the best model (#10) is actually wrong, using out-of-fold predictions on
the training split only — the final test set is never loaded here.

**Best model per `results.md`:** RandomForest(n_estimators=200,
min_samples_leaf=50), 37 columns (all `ps_calc_*` dropped), Gini
0.27244 ± 0.00305, commit `6c794f2`. #17's later attempt to also drop the
individually-dead columns was a **REJECT** (its gain didn't clear its own
fold std), so this 37-column model is still the one on top.

**Out-of-fold, not the holdout:** `cross_val_predict` with the *same*
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` used by
`evaluation.cross_validate_model` (#8) — each row is scored by the one fold
model that never trained on it, so the numbers below are honest even though
every row gets used.

In [1]:
# Make the notebook behave as if it were running from the repo root.
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

from data_loading import feature_groups
from evaluation import SEED, N_FOLDS, gini_normalized, load_train
from train_baseline_models import _build_preprocessor

# load_final_test is never imported or called anywhere in this notebook (AC-7).
train = load_train()
categorical_cols, quantity_cols = feature_groups(train.columns)

# The 37-column best model from results.md / #13: all ps_calc_* dropped.
cat_nocalc = [c for c in categorical_cols if "_calc_" not in c]
qty_nocalc = [c for c in quantity_cols if "_calc_" not in c]

X = train[cat_nocalc + qty_nocalc]
y = train["target"]
print(f"rows {len(train):,} | {len(cat_nocalc)} categorical + {len(qty_nocalc)} quantity "
      f"= {len(cat_nocalc) + len(qty_nocalc)} columns")

rows 475,967 | 25 categorical + 12 quantity = 37 columns


## AC-1 — out-of-fold predictions, whole training split

Same model, same folds as the recorded 0.27244 ± 0.00305 result — this is a
reproduction with per-row predictions kept, not a new model.

In [2]:
model = make_pipeline(
    _build_preprocessor(len(cat_nocalc), len(qty_nocalc), scale=False),
    RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                           n_jobs=-1, random_state=SEED),
)
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_risk = cross_val_predict(model, X, y, cv=splitter, method="predict_proba")[:, 1]
train = train.assign(oof_risk=oof_risk)

pooled_gini = gini_normalized(y, oof_risk)
print(f"pooled OOF gini: {pooled_gini:.5f}")
print(f"reference (mean of 5 fold ginis, results.md): 0.27244 +/- 0.00305")
print(f"\nsanity check: pooling every row's OOF prediction into one score "
      f"reproduces the recorded result (folds average to the same place a "
      f"single pooled ranking does when fold sizes are close to equal).")
assert abs(pooled_gini - 0.27244) < 0.00305, "OOF reproduction drifted outside the recorded fold std"
print("PASS")

pooled OOF gini: 0.27243
reference (mean of 5 fold ginis, results.md): 0.27244 +/- 0.00305

sanity check: pooling every row's OOF prediction into one score reproduces the recorded result (folds average to the same place a single pooled ranking does when fold sizes are close to equal).
PASS


## AC-3 — share of actual claims in the riskiest 10% of predictions

A model with real ranking power should pack more than 10% of the actual
claims into the top 10% of its own risk scores. This is the plainest way to
say how much the ranking actually buys us.

In [3]:
n_top = int(len(train) * 0.10)
top_decile = train.nlargest(n_top, "oof_risk")

total_claims = train["target"].sum()
claims_in_top = top_decile["target"].sum()
capture = claims_in_top / total_claims

print(f"riskiest 10% of rows: {n_top:,}")
print(f"claims in that slice: {claims_in_top:,} of {total_claims:,} total claims")
print(f"capture rate: {capture*100:.2f}%  (10.00% is what a random ranking would capture)")
print(f"lift over random: {capture/0.10:.2f}x")

riskiest 10% of rows: 47,596
claims in that slice: 3,683 of 17,461 total claims
capture rate: 21.09%  (10.00% is what a random ranking would capture)
lift over random: 2.11x


## AC-2 — actual claim rate vs. predicted risk across three slices

Three slices, each pointed at by a specific finding from #13 rather than
picked at random:

- **A — missing values per row.** #13's shortlist item 2 flagged missingness
  as informative (`ps_reg_03`'s missing rows claim at a different rate than
  the base). Does that show up in how well-calibrated the model is as a row
  accumulates more missing fields?
- **B — `ps_car_04_cat` rare vs. common levels.** #13's method note singled
  this column out: highest univariate lift spread of the top ten (1.381),
  yet ranked only 10th by permutation importance — a column the model
  under-uses relative to what it separates on its own.
- **C — one representative feature from each group #13's ablation proved
  indispensable** (`reg` −0.01378, `car` −0.03667, `ind` −0.05751 — dropping
  any of them cost real Gini). Each group's top permutation-importance
  column stands in for the group: `ps_ind_05_cat`, `ps_reg_03`, `ps_car_13`.

In [4]:
def slice_table(group_series):
    g = train.groupby(group_series, observed=True)
    out = g.agg(n=("target", "size"), claims=("target", "sum"),
               actual_rate=("target", "mean"), mean_pred=("oof_risk", "mean"))
    out["pct_of_rows"] = (out.n / len(train) * 100).round(2)
    out["actual_rate_pct"] = (out.actual_rate * 100).round(4)
    out["mean_pred_pct"] = (out.mean_pred * 100).round(4)
    out["gap_pp"] = (out.actual_rate_pct - out.mean_pred_pct).round(4)
    return out[["n", "claims", "pct_of_rows", "actual_rate_pct", "mean_pred_pct", "gap_pp"]]


def decile_or_level(col, n_bins=5):
    """Categorical -> its own raw levels (missing gets its own row).
    Quantity -> quantiles, missing (-1) split out first."""
    if col.endswith(("_cat", "_bin")):
        s = train[col].astype(str)
        return s.where(train[col] != -1, "missing")
    missing = train[col] == -1
    g = pd.Series(index=train.index, dtype=object)
    g[missing] = "missing"
    g[~missing] = "q" + (pd.qcut(train.loc[~missing, col], n_bins,
                                 labels=False, duplicates="drop") + 1).astype(str)
    return g

In [5]:
# Slice A -- missing values per row
feature_cols = cat_nocalc + qty_nocalc  # the 37 columns the model actually sees
n_missing = (train[feature_cols] == -1).sum(axis=1)
bucket_a = pd.cut(n_missing, [-1, 0, 1, 2, 100], labels=["0", "1", "2", "3+"])
print("Slice A -- missing values per row (of the 37 columns the model uses)")
slice_table(bucket_a)

Slice A -- missing values per row (of the 37 columns the model uses)


,n,claims,pct_of_rows,actual_rate_pct,mean_pred_pct,gap_pp
0,99867,4554,20.98,4.5601,4.4495,0.1106
1,124943,4590,26.25,3.6737,3.7339,-0.0602
2,203944,6976,42.85,3.4205,3.4086,0.0119
3+,47213,1341,9.92,2.8403,2.9910,-0.1507


In [6]:
# Slice B -- ps_car_04_cat rare vs. common levels
rare_levels = {3, 4, 5, 6, 7}  # the 5 levels with < 1,300 rows each; see EDA below
level_counts = train["ps_car_04_cat"].value_counts()
print("ps_car_04_cat row counts per level:")
print(level_counts.sort_index())

bucket_b = train["ps_car_04_cat"].map(lambda v: "rare" if v in rare_levels else "common")
print("\nSlice B -- ps_car_04_cat rare vs. common levels")
slice_table(bucket_b)

ps_car_04_cat row counts per level:
ps_car_04_cat
0    397106
1     25664
2     18981
3       515
4       187
5       458
6      1260
7       114
8     16461
9     15221
Name: count, dtype: int64

Slice B -- ps_car_04_cat rare vs. common levels


,n,claims,pct_of_rows,actual_rate_pct,mean_pred_pct,gap_pp
ps_car_04_cat,,,,,,
common,473433,17324,99.47,3.6592,3.6618,-0.0026
rare,2534,137,0.53,5.4065,5.3785,0.0280


In [7]:
# Slice C -- one representative column per group #13's ablation proved indispensable
for col in ["ps_ind_05_cat", "ps_reg_03", "ps_car_13"]:
    print(f"\nSlice C -- {col}")
    display(slice_table(decile_or_level(col)))


Slice C -- ps_ind_05_cat


,n,claims,pct_of_rows,actual_rate_pct,mean_pred_pct,gap_pp
ps_ind_05_cat,,,,,,
0,422193,14408,88.70,3.4127,3.4865,-0.0738
1,6706,328,1.41,4.8911,4.6879,0.2032
2,3322,241,0.70,7.2547,5.2821,1.9726
3,6631,293,1.39,4.4186,4.5611,-0.1425
4,14642,768,3.08,5.2452,4.7954,0.4498
5,1323,64,0.28,4.8375,4.7603,0.0772
6,16479,977,3.46,5.9288,5.1874,0.7414
missing,4671,382,0.98,8.1781,7.2957,0.8824



Slice C -- ps_reg_03


,n,claims,pct_of_rows,actual_rate_pct,mean_pred_pct,gap_pp
missing,86133,2464,18.10,2.8607,3.0651,-0.2044
q1,78224,2293,16.43,2.9313,3.1124,-0.1811
q2,77753,2640,16.34,3.3954,3.2942,0.1012
q3,77951,2934,16.38,3.7639,3.6626,0.1013
q4,77965,3404,16.38,4.3661,4.2010,0.1651
q5,77941,3726,16.38,4.7805,4.7553,0.0252



Slice C -- ps_car_13


,n,claims,pct_of_rows,actual_rate_pct,mean_pred_pct,gap_pp
q1,95199,2412,20.0,2.5336,2.6873,-0.1537
q2,95188,2970,20.0,3.1201,3.0621,0.0580
q3,95196,3254,20.0,3.4182,3.4430,-0.0248
q4,95190,3803,20.0,3.9952,4.0004,-0.0052
q5,95194,5022,20.0,5.2755,5.1622,0.1133


## Findings

**Slices A and C (`ps_reg_03`, `ps_car_13`) and the common levels of B are
all well-calibrated** — actual-vs-predicted gaps stay under ~0.2 percentage
points against base rates around 3-5%. No single slice there explains a
meaningful share of the model's error; that is itself consistent with what
#13's own notes predicted for a model at this Gini level.

**One slice breaks that pattern: `ps_ind_05_cat`'s minority levels.**
`ps_ind_05_cat` is the single most-relied-on feature by permutation
importance (#13), yet the model *under-predicts* risk for every one of its
minority values, in the same direction every time:

| level | rows | actual rate | predicted | gap |
|---|---|---|---|---|
| 0 (88.7% of rows) | 422,193 | 3.41% | 3.49% | −0.07pp |
| 2 | 3,322 | 7.25% | 5.28% | **+1.97pp** |
| missing | 4,671 | 8.18% | 7.30% | +0.88pp |
| 6 | 16,479 | 5.93% | 5.19% | +0.74pp |
| 4 | 14,642 | 5.25% | 4.80% | +0.45pp |

Level 2 is the sharpest case: 3,322 rows (0.7% of the training split) with a
true claim rate of 7.25%, standard error ≈0.45pp — the model's 5.28% average
prediction there misses by roughly 4 standard errors, understating that
group's risk by more than a quarter. Every minority level moves the same
way; only the 88.7%-share modal level (0) is priced correctly.

## AC-4 — Limitation

**The model systematically under-prices the minority levels of
`ps_ind_05_cat`, its own most-relied-on feature**, while pricing the modal
level correctly. Evidence: the table above — four of `ps_ind_05_cat`'s seven
non-modal levels (2, 4, 6, and missing) each show a positive, non-noise gap
between actual and predicted risk, level 2 being the largest at +1.97
percentage points against a predicted 5.28% (n=3,322, gap ≈4× the level's own
standard error). `min_samples_leaf=50` forces every tree leaf to pool at
least 50 rows, and with only 0.7-3.5% of rows per minority level, leaves that
would otherwise isolate a minority level get merged with neighbouring,
lower-risk rows — diluting the minority signal toward the majority rate. The
features are anonymized, so nothing here says *why* `ps_ind_05_cat`
separates risk this way, only that it does and that the current model
underuses it exactly where it matters most.

## AC-5 — Recommendation

Because this is a leaf-pooling effect on a specific known column, not a
diffuse problem, the fix is targeted rather than a general model change:
**replace `ps_ind_05_cat`'s one-hot encoding with target (mean) encoding**,
computed out-of-fold to avoid leakage. That gives each minority level its
own informed base rate as a single numeric input instead of relying on
`min_samples_leaf=50` to keep it separable inside a shared tree structure —
directly addressing the mechanism identified above. This was already #13's
shortlist item 4 (there proposed for `ps_car_11_cat`'s width, 104 levels);
this analysis adds a second, independent reason to run it, motivated by
calibration rather than cost, and widens the candidate column list to
include `ps_ind_05_cat`.

## AC-6 — What we would try next, ordered by expected value

1. **Target-encode `ps_ind_05_cat` (and `ps_car_11_cat`).** Highest expected
   value: it's a direct fix for the one concrete, evidenced miscalibration
   found above, re-uses #13's already-scoped idea, and is cheap to test
   through the existing harness (#8).
2. **Add `<col>_was_missing` flags for `ps_reg_03` / `ps_car_14`** (#13
   shortlist item 2). Slice A here shows missingness-count calibration is
   already close (gaps <0.2pp), so the expected gain is smaller than item 1,
   but it is still the next-cheapest untested lever on #13's list.
3. **Revisit `min_samples_leaf`** for `ps_ind_05_cat` specifically (e.g. a
   lower leaf minimum, or a second forest specialised on rows with a
   minority `ps_ind_05_cat` value). Directly targets the leaf-pooling
   mechanism named in AC-4, but touches model structure rather than data, so
   it's riskier to reason about and slower to validate than encoding.
4. **Try isotonic or Platt calibration on top of the existing forest.**
   Wouldn't change ranking (Gini) but could tighten the actual-vs-predicted
   gaps seen in slice C's tails without retraining anything — worth trying
   once a target-encoded version exists to compare against.
5. **With real project time (not this course):** ask Porto Seguro's data
   team what `ps_ind_05_cat` encodes. Anonymization is a hard floor on how
   far this analysis can go — recommendation 1 fixes the *symptom* the model
   shows; only de-anonymized business context could confirm the *cause*.